<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="50%"></a>
</p>


# Interfacing with IMU

The inertial measurement unit (IMU) data that [Notebook 5](./5-duckiedrone-imu-architecture.ipynb) and [Notebook 6](./6-imu-rates-and-configuration.ipynb) followed ends up on a ROS 2 topic. This notebook reads it, first with ROS 2 tools in a terminal, then with a short node that prints roll, pitch and yaw every second while the Duckiedrone is tilted.

PX4 and MAVROS2 have already calibrated the measurements, estimated the orientation and converted both to ROS conventions, so the node stays short. What remains is to ask ROS 2 for the messages and to unpack them.

By the end of the notebook you should be able to answer:

- Which fields of `/mavros/imu/data` are sensor measurements, and which one is an estimate?

- Why does pitch stay between $-90^\circ$ and $+90^\circ$ while roll covers the full circle?

- Why can yaw drift while roll and pitch stay correct?

## Getting to a terminal that can see the Duckiedrone

The topics used below exist on the Duckiedrone itself, on the Raspberry Pi side of the MAVLink arrow in [Figure 1 of Notebook 5](./5-duckiedrone-imu-architecture.ipynb#figure-1).

On a physical Duckiedrone:

```bash
ssh duckie@DUCKIEDRONE_NAME.local
```

For a virtual Duckiedrone:

```bash
dts duckiebot virtual connect DUCKIEDRONE_NAME
```

On a physical Duckiedrone, the MAVROS2 service is normally `ros2-mavros`:

```bash
docker ps --format '{{.Names}}'
docker exec -it ros2-mavros bash
```

`ros2-mavros` holds the flight-controller link and publishes `/mavros/imu/data`, as [the PX4-to-MAVROS2 section of Notebook 5](./5-duckiedrone-imu-architecture.ipynb#from-px4-to-mavros2-over-mavlink) shows. There is no separate IMU driver container because the IMU is wired to the flight controller, not to the Duckiedrone computer. On a virtual Duckiedrone, use `docker ps` to find the MAVROS2 container if its name differs. Start the Duckiematrix and attach the Duckiedrone to an entity before expecting simulated IMU data.

## Look at the data first

Inspect the IMU data path before writing a line of code.

First, list the IMU-related topics:

```bash
ros2 topic list | grep imu
```

This should list `/mavros/imu/data` and `/mavros/imu/data_raw`, among a few others. If the names come back with a `/DUCKIEDRONE_NAME` prefix in front, the Duckiedrone publishes under a namespace and every command below needs that prefix too.

Next, measure the publication rate:

```bash
ros2 topic hz /mavros/imu/data
```

The MAVLink stream is configured for $50\,\mathrm{Hz}$, and a physical Duckiedrone measures about that rate. Timing on a virtual Duckiedrone can differ, so measure it rather than assuming the configured value.

Finally, print one message:

```bash
ros2 topic echo --once /mavros/imu/data
```

In what `echo` prints, `orientation` is a quaternion, `angular_velocity` is in $\mathrm{rad/s}$, and `linear_acceleration` is in $\mathrm{m/s^2}$. Despite its name, `linear_acceleration` holds the accelerometer's proper-acceleration measurement ([Notebook 5](./5-duckiedrone-imu-architecture.ipynb#from-px4-to-mavros2-over-mavlink)), so a level, stationary Duckiedrone shows a $z$ component near $+9.81\,\mathrm{m/s^2}$ rather than zero. Each group has a covariance array that describes uncertainty when the publisher provides it. In a `sensor_msgs/Imu` message, an all-zero covariance means the covariance is unknown; for `orientation_covariance`, a first entry of `-1` means no orientation estimate is available.

Run the same `echo` on `/mavros/imu/data_raw` and compare. Its orientation is typically zeros with `orientation_covariance[0]` set to `-1`, because that topic carries the accelerometer and gyroscope without the fused attitude.

## Activity 1: write the subscriber

<a id="activity-1"></a>

Open [packages/solution/solution/imu_subscriber.py](../packages/solution/solution/imu_subscriber.py). Most of the file is already written: it creates the node, works out the topic name from the `robot_namespace` parameter, sets `self.latest_reading` to `None`, and starts a timer that calls `cb_log` every second to print.

Two pieces are missing. Each is marked in the file with a `TO IMPLEMENT` block listing the names and expressions to use.

### In `__init__`, subscribe to the IMU topic

Create the subscription with `Imu`, the `topic` variable, `self.cb_imu`, and `qos_profile_sensor_data`, then store the returned subscription in `self.sub_imu`.

`qos_profile_sensor_data` matches the MAVROS2 IMU publisher's best-effort sensor-data profile with a queue depth of 5. Inspect the publisher on the Duckiedrone to confirm it:

```bash
ros2 topic info -v /mavros/imu/data
```

### In `cb_imu`, unpack one message

The callback receives one `sensor_msgs/Imu` and stores an `ImuReading` in `self.latest_reading`, where `cb_log` finds it a second later. Set its timestamp from `msg.header.stamp`, its acceleration from `msg.linear_acceleration`, and its gyroscope reading from `msg.angular_velocity`. Convert `msg.orientation` from the MAVROS2 quaternion to roll, pitch, and yaw with `quaternion_to_euler`.

Of the three groups in the message, only two are sensor measurements: the calibrated proper acceleration and angular velocity. `orientation` is the flight controller's fused attitude estimate, computed from those measurements and earlier estimator data.

The callback stores rather than prints the reading: the topic arrives at about $50\,\mathrm{Hz}$, while `cb_log` prints one human-readable line per second. With both pieces written, the node is ready to run.

## Activity 2: run the node on the Duckiedrone

<a id="activity-2"></a>

From the project's root folder, build and run the node:

```bash
dts code build -R DUCKIEDRONE_NAME
dts code workbench -R DUCKIEDRONE_NAME
```

The node's output appears in the terminal every second:

```shell
[imu_listener_node-1] [INFO] [imu_listener_node]: accel=[-0.07 -0.11 +9.75] m/s^2 | gyro=[+0.00 +0.00 -0.00] rad/s | rpy=[  -0.6   +0.4  +89.9] deg
```

After changing the code, repeat the same build-and-workbench sequence.

### Running the launch file with a namespace

The default launcher uses no namespace. If the Duckiedrone publishes IMU data under `/DUCKIEDRONE_NAME`, start a second listener in a workbench shell with:

```bash
ros2 launch imu_listener imu_listener_node.launch.py robot_namespace:=DUCKIEDRONE_NAME
```

The node logs its subscribed topic at startup. Compare it with the topic listed above to distinguish a namespace mismatch from an upstream Duckiedrone problem.

### When nothing arrives

For this IMU path, check these sensor-specific points:

- `ros2 node list` should show MAVROS2 as `/mavros`, or `/DUCKIEDRONE_NAME/mavros` when the Duckiedrone
  uses that namespace.

- The topic logged at startup must match a topic in `ros2 topic list`.

- `ros2 topic hz /mavros/imu/data` should report about $50\,\mathrm{Hz}$ on a physical Duckiedrone.
  Use `/DUCKIEDRONE_NAME/mavros/imu/data` when the node logged that namespaced topic. Timing can differ on a
  virtual Duckiedrone; silence there points upstream of the listener, on the flight-controller link.

## Activity 3: tilt the Duckiedrone and watch the angles

<a id="activity-3"></a>

On a physical Duckiedrone, do this only while it is disarmed, and keep clear of the propellers.

Leave the listener running and tilt the Duckiedrone in pitch and roll, using the Duckiematrix controls on a virtual Duckiedrone or by hand on a physical one. Check the listener output:

- Roll follows the applied tilt and stays between $-180^\circ$ and $+180^\circ$.

- Pitch follows too, but stays between $-90^\circ$ and $+90^\circ$. This half range is the ZYX Euler convention from [Notebook 3](./3-rotation-representations.ipynb), not a bug in the code.

- `accel[2]`, the listener's display of `linear_acceleration.z`, sits near $+9.81\,\mathrm{m/s^2}$ when the Duckiedrone is level and still, and falls as it tilts, because the same proper-acceleration measurement is then shared across all three axes.

- Yaw has no absolute reference. While the Duckiedrone is still or moving gently, this proper-acceleration measurement lets the estimator correct roll and pitch. Nothing outside the Duckiedrone points north, so yaw starts from an arbitrary heading and can drift.

Then set the Duckiedrone down, leave it still for a minute, and watch yaw. Note whether it creeps rather than jumps. Slow drift is expected when a bias in the measured angular velocity is integrated, but the amount depends on the estimator and the sensor. A virtual Duckiedrone's simulated gyroscope may not model the same bias, so yaw there may not drift at all.

## Going further

Each of these is optional:

- Subscribe to `/mavros/imu/data_raw` alongside `/mavros/imu/data` and print both. Compare the
  acceleration, angular-velocity, orientation, and covariance fields. Which values differ, and
  which topic contains the flight controller's fused orientation?

- Record the IMU while tilting the Duckiedrone, then plot roll, pitch, and yaw.  `ros2 bag record /mavros/imu/data` writes every message on the topic to a file that can be replayed or exported afterwards. The sample recording supplied with this LX was captured this way and then converted to the CSV that ships with this LX.

## Further reading

The [sensor_msgs/Imu message definition](https://github.com/ros2/common_interfaces/blob/jazzy/sensor_msgs/msg/Imu.msg) lists the fields unpacked by the subscriber.

## Checkpoint

Run the self-check in the next cell. Write or select a response before revealing the answer.


In [ ]:
import sys
from pathlib import Path

working_directory = Path.cwd()
parent_directory = working_directory.parent
if (parent_directory / "packages").is_dir():
    parent_directory_path = str(parent_directory)
    sys.path.insert(0, parent_directory_path)

from packages.checkpoint_self_check import display_checkpoint_self_checks

display_checkpoint_self_checks()